In [ ]:
# === 分數分佈校準 (修復 KeyError: L1/L2/L3) ===
import numpy as np
import matplotlib.pyplot as plt

# 1. 定義鍵值，對齊使用者現有的 L1, L2, L3 邏輯
layers = ['L1', 'L2', 'L3']
layer_names = {'L1': '聲紋韻律', 'L2': 'Deepfake', 'L3': '語義詐騙'}
real_scores = {l: [] for l in layers}
fake_scores = {l: [] for l in layers}

# 2. 生成模擬校準數據
for _ in range(1000):
    # L1: 韻律
    real_scores['L1'].append(np.random.beta(2, 5))
    fake_scores['L1'].append(np.random.beta(5, 2))
    
    # L2: Deepfake
    real_scores['L2'].append(np.random.beta(2, 8))
    fake_scores['L2'].append(np.random.beta(8, 2))
    
    # L3: 語義
    real_scores['L3'].append(np.random.beta(1, 10))
    fake_scores['L3'].append(np.random.beta(7, 2))

print('✅ 校準數據已對齊 (Keys: L1, L2, L3)')

# 3. 視覺化區塊
plt.figure(figsize=(15, 4))
for i, layer in enumerate(layers):
    plt.subplot(1, 3, i+1)
    plt.hist(real_scores[layer], bins=30, alpha=0.5, label='真人', color='#4ade80')
    plt.hist(fake_scores[layer], bins=30, alpha=0.5, label='詐騙', color='#f87171')
    plt.title(f'{layer_names[layer]} ({layer}) 分數分佈')
    plt.legend()
plt.tight_layout()
plt.show()

# 🧠 Notebook 3：記憶庫冷啟動 + SE-Attention 融合引擎訓練

**專案**: AI_Voice 智慧語音詐騙檢測工具  
**目標**:  
- Part A：使用 ChiFraud 建立 FAISS 初始記憶庫  
- Part B：使用合成的 Agent 輸出訓練 SE-Attention MLP 融合模型  
**平台**: Kaggle GPU (T4 x2)  
**輸出**: `faiss.index` + `metadata.json` + `se_attention_mlp.pt`

In [ ]:
# === Matplotlib 中文顯示修復 (手動路徑版) ===
!apt-get install -y fonts-wqy-microhei
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 手動強制加載字型檔，避開快取更新問題
font_path = '/usr/share/fonts/truetype/wqy/wqy-microhei.ttc'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
    plt.rcParams['axes.unicode_minus'] = False
    print('✅ 已手動載入字型: WenQuanYi Micro Hei')
else:
    print('❌ 找不到字型檔，請確認已執行 apt-get install')


In [ ]:
!pip install -q praat-parselmouth -q faiss-cpu sentence-transformers opencc-python-reimplemented

import os, json, time, glob, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import faiss
from sentence_transformers import SentenceTransformer
import opencc
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'裝置: {device}')

---
## Part A：FAISS 記憶庫冷啟動

從 ChiFraud 詐騙子集中選取代表性案例，
使用 Sentence-BERT 生成語義嵌入並建立 FAISS 索引。

In [ ]:
# === 下載並載入 ChiFraud（Tab 分隔，Label_id + Text）===
!git clone https://github.com/xuemingxxx/ChiFraud.git /tmp/ChiFraud 2>/dev/null || echo '已存在'

csv_files = sorted(glob.glob('/tmp/ChiFraud/dataset/*.csv'))
print(f'CSV 檔案: {csv_files}')

dfs = []
for f in csv_files:
    try:
        tmp = pd.read_csv(f, sep='\t', names=['Label_id', 'Text'],
                          header=None, on_bad_lines='skip',
                          encoding='utf-8', engine='python')
        tmp = tmp[tmp['Label_id'].apply(lambda x: str(x).strip().isdigit())]
        tmp['Label_id'] = tmp['Label_id'].astype(int)
        print(f'  {os.path.basename(f)}: {len(tmp)} 筆')
        dfs.append(tmp)
    except Exception as e:
        print(f'  {os.path.basename(f)} 失敗: {e}')

df = pd.concat(dfs, ignore_index=True)
print(f'\n總計: {len(df)} 筆')

In [ ]:
# === 篩選詐騙案例 (Label_id != 0) ===
FRAUD_CATEGORIES = {
    1: '賭博', 2: '色情', 3: '假證件', 4: '假銀行卡',
    5: '違禁藥物', 6: '非法套現', 7: '非法認證',
    8: '假SIM卡', 9: '地下貸款', 10: '新型詐騙'
}

fraud_df = df[df['Label_id'] != 0].copy()
print(f'詐騙案例: {len(fraud_df)} 筆')

# 繁簡轉換 + 清理
converter = opencc.OpenCC('s2twp')
def clean_convert(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'[【\[].{0,20}[】\]]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return converter.convert(text)

fraud_df['text_tw'] = fraud_df['Text'].apply(clean_convert)
fraud_df = fraud_df[fraud_df['text_tw'].str.len() > 10].drop_duplicates(subset='text_tw')

# 冷啟動限制 5000 筆
MAX_CASES = 5000
if len(fraud_df) > MAX_CASES:
    fraud_df = fraud_df.sample(n=MAX_CASES, random_state=42)

print(f'冷啟動案例: {len(fraud_df)} 筆')
fraud_df['text_tw'].head(3)

In [ ]:
# === 生成語義嵌入 ===
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
texts = fraud_df['text_tw'].tolist()
print(f'正在生成 {len(texts)} 個嵌入向量...')

embeddings = embed_model.encode(texts, show_progress_bar=True,
                                 batch_size=128, normalize_embeddings=True)
embeddings = np.array(embeddings, dtype=np.float32)
print(f'嵌入矩陣: {embeddings.shape}')

In [ ]:
# === 建立 FAISS 索引 + 元資料 ===
EMBEDDING_DIM = embeddings.shape[1]
index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(embeddings)
print(f'FAISS 索引: {index.ntotal} 向量, 維度 {EMBEDDING_DIM}')

metadata = []
for _, row in fraud_df.iterrows():
    label_id = int(row['Label_id'])
    fraud_type = FRAUD_CATEGORIES.get(label_id, '未分類')
    metadata.append({
        'text': row['text_tw'][:200],
        'fraud_type': fraud_type,
        'confirmed': True,
        'timestamp': '2026-01-01 00:00:00',
        'source': 'ChiFraud',
    })

print(f'\n詐騙類型分布:')
print(pd.Series([m['fraud_type'] for m in metadata]).value_counts())

In [ ]:
# === 搜尋驗證 ===
for query in ['你好，這裡是公安局，你的帳戶涉嫌洗錢',
              '恭喜您中了五百萬大獎，請先匯手續費',
              '你的快遞包裹已到達，請確認收貨地址']:
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(q_emb.astype(np.float32), 3)
    print(f'\n查詢:「{query}」')
    for s, i in zip(scores[0], indices[0]):
        m = metadata[i]
        print(f'  [{s:.3f}] [{m["fraud_type"]}] {m["text"][:40]}...')

In [ ]:
# === 儲存 ===
os.makedirs('output/memory', exist_ok=True)
faiss.write_index(index, 'output/memory/faiss.index')
with open('output/memory/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f'已儲存! {len(metadata)} 筆')

---
## Part B：SE-Attention MLP 融合引擎訓練

```
輸入: [P_v, C_v, Q_v, P_s, C_s, Q_s, P_m, C_m, Q_m]  (9維)
    → FC(9→32, ReLU) → FC(32→16, ReLU) → FC(16→3, Softmax)
    → [w_v, w_s, w_m]
```

In [ ]:
class SEAttentionMLP(nn.Module):
    def __init__(self, n_agents=3, features_per_agent=3):
        super().__init__()
        self.excitation = nn.Sequential(
            nn.Linear(n_agents * features_per_agent, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(16, n_agents),
        )
    def forward(self, x):
        return self.excitation(x)

se_model = SEAttentionMLP().to(device)
print(f'參數量: {sum(p.numel() for p in se_model.parameters()):,}')

In [ ]:
# === 合成訓練資料（6 種場景）===
np.random.seed(42)
N = 10000
X_data, y_data = [], []

for _ in range(N):
    s = np.random.choice(['balanced','voice_dom','sem_dom','mem_dom','low_q','all_high'])
    if s == 'balanced':
        v = [np.random.uniform(.3,.7), np.random.uniform(.5,.9), np.random.uniform(.5,.9)]
        se = [np.random.uniform(.3,.7), np.random.uniform(.5,.9), np.random.uniform(.5,.9)]
        m = [np.random.uniform(.3,.7), np.random.uniform(.5,.9), np.random.uniform(.5,.9)]
        w = [0.35, 0.40, 0.25]
    elif s == 'voice_dom':
        v = [np.random.uniform(.5,.95), np.random.uniform(.7,.95), np.random.uniform(.8,1)]
        se = [np.random.uniform(.1,.5), np.random.uniform(.3,.6), np.random.uniform(.2,.5)]
        m = [np.random.uniform(0,.3), np.random.uniform(.3,.5), np.random.uniform(.2,.4)]
        w = [0.60, 0.25, 0.15]
    elif s == 'sem_dom':
        v = [np.random.uniform(.1,.4), np.random.uniform(.3,.5), np.random.uniform(.3,.5)]
        se = [np.random.uniform(.6,.95), np.random.uniform(.7,.95), np.random.uniform(.8,1)]
        m = [np.random.uniform(.1,.4), np.random.uniform(.3,.6), np.random.uniform(.3,.6)]
        w = [0.15, 0.60, 0.25]
    elif s == 'mem_dom':
        v = [np.random.uniform(.1,.4), np.random.uniform(.3,.6), np.random.uniform(.3,.6)]
        se = [np.random.uniform(.2,.5), np.random.uniform(.4,.7), np.random.uniform(.4,.7)]
        m = [np.random.uniform(.7,.95), np.random.uniform(.8,.95), np.random.uniform(.8,1)]
        w = [0.15, 0.25, 0.60]
    elif s == 'low_q':
        v = [np.random.uniform(0,.3), np.random.uniform(.1,.3), np.random.uniform(0,.2)]
        se = [np.random.uniform(0,.3), np.random.uniform(.1,.3), np.random.uniform(0,.2)]
        m = [np.random.uniform(0,.3), np.random.uniform(.1,.3), np.random.uniform(0,.2)]
        w = [0.33, 0.34, 0.33]
    else:
        v = [np.random.uniform(.7,.95), np.random.uniform(.8,.95), np.random.uniform(.8,1)]
        se = [np.random.uniform(.7,.95), np.random.uniform(.8,.95), np.random.uniform(.8,1)]
        m = [np.random.uniform(.7,.95), np.random.uniform(.8,.95), np.random.uniform(.8,1)]
        w = [0.35, 0.40, 0.25]
    w = np.array(w) + np.random.randn(3)*0.05
    w = np.clip(w, 0.05, 0.95); w = w / w.sum()
    X_data.append(v + se + m); y_data.append(w.tolist())

X_data = np.array(X_data, dtype=np.float32)
y_data = np.array(y_data, dtype=np.float32)
print(f'X={X_data.shape}, y={y_data.shape}')

In [ ]:
# === 訓練 ===
from torch.utils.data import TensorDataset, DataLoader
split = int(0.85 * len(X_data))
train_X, test_X = torch.tensor(X_data[:split]).to(device), torch.tensor(X_data[split:]).to(device)
train_y, test_y = torch.tensor(y_data[:split]).to(device), torch.tensor(y_data[split:]).to(device)
loader = DataLoader(TensorDataset(train_X, train_y), batch_size=128, shuffle=True)

opt = torch.optim.AdamW(se_model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
kl = nn.KLDivLoss(reduction='batchmean')

best, tl, vl = float('inf'), [], []
os.makedirs('output', exist_ok=True)
for ep in range(50):
    se_model.train(); el = 0
    for bx, by in loader:
        opt.zero_grad(); loss = kl(torch.log_softmax(se_model(bx), -1), by)
        loss.backward(); opt.step(); el += loss.item()
    sched.step(); tl.append(el/len(loader))
    se_model.eval()
    with torch.no_grad():
        v = kl(torch.log_softmax(se_model(test_X), -1), test_y).item(); vl.append(v)
    if v < best: best = v; torch.save(se_model.state_dict(), 'output/se_attention_mlp.pt')
    if (ep+1) % 10 == 0: print(f'Ep {ep+1}/50 | T:{tl[-1]:.6f} V:{v:.6f} B:{best:.6f}')
print(f'\n完成! Best KL: {best:.6f}')

In [ ]:
# === 訓練曲線 ===
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(tl, label='Train', color='#00f2ff'); ax.plot(vl, label='Val', color='#ff0055')
ax.set_xlabel('Epoch'); ax.set_ylabel('KL Divergence'); ax.set_title('SE-Attention 訓練曲線')
ax.legend(); ax.set_facecolor('#0f172a'); fig.patch.set_facecolor('#0f172a')
ax.tick_params(colors='white'); ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white'); ax.title.set_color('white')
plt.tight_layout(); plt.savefig('se_training.png', dpi=150); plt.show()

In [ ]:
# === 推理測試 ===
se_model.load_state_dict(torch.load('output/se_attention_mlp.pt'))
se_model.eval()
for name, feat in [('聲紋強勢',[.85,.9,.95,.3,.4,.3,.1,.3,.2]),
                    ('語義強勢',[.2,.4,.3,.9,.9,.95,.2,.4,.4]),
                    ('記憶強勢',[.2,.4,.4,.3,.5,.5,.9,.9,.95]),
                    ('均衡高品質',[.8,.85,.9,.8,.85,.9,.8,.85,.9]),
                    ('低品質',[.1,.2,.1,.1,.2,.1,.1,.2,.1])]:
    w = torch.softmax(se_model(torch.tensor([feat],dtype=torch.float32).to(device)),dim=-1).squeeze().cpu().numpy()
    print(f'{name:>8} → V:{w[0]:.1%} S:{w[1]:.1%} M:{w[2]:.1%}')

In [ ]:
!tar -czf memory_and_fusion.tar.gz -C output .
print('打包完成！下載 memory_and_fusion.tar.gz')
print('  memory/faiss.index   → AI_Voice/models/memory/')
print('  memory/metadata.json → AI_Voice/models/memory/')
print('  se_attention_mlp.pt  → AI_Voice/models/fusion/')